# Day 17 — Feature engineering: encoding, scaling and derived features

Run each cell with **Shift + Enter**, from the top. We use `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Introduction: why feature engineering matters

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-017/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import pandas as pd

# The lesson's five students: raw features
df = pd.DataFrame({"study_hours": [2, 4, 6, 3, 8], "attendance": [55, 70, 90, 65, 95],
                   "prev_score": [40, 55, 85, 50, 92], "assignments_done": [3, 5, 9, 4, 10]})
print(df.to_string(index=False))

 study_hours  attendance  prev_score  assignments_done
           2          55          40                 3
           4          70          55                 5
           6          90          85                 9
           3          65          50                 4
           8          95          92                10


In [ ]:
# Feature creation and binning
df["weekly_study_hours"] = df["study_hours"] * 7
df["attendance_category"] = pd.cut(df["attendance"], bins=[0, 60, 80, 100], labels=["Low", "Medium", "High"], include_lowest=True)
df["score_category"] = pd.cut(df["prev_score"], bins=[0, 50, 70, 100], labels=["Low", "Medium", "High"], include_lowest=True)
df["assignment_completion"] = df["assignments_done"] / 10          # completion ratio, assuming 10 assignments were set
print(df.to_string(index=False))

 study_hours  attendance  prev_score  assignments_done  weekly_study_hours attendance_category score_category  assignment_completion
           2          55          40                 3                  14                 Low            Low                    0.3
           4          70          55                 5                  28              Medium         Medium                    0.5
           6          90          85                 9                  42                High           High                    0.9
           3          65          50                 4                  21              Medium            Low                    0.4
           8          95          92                10                  56                High           High                    1.0


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

# Model A vs Model B on our 127 training students: does engineering features help? (pass/fail, 5-fold CV)
data = pd.read_csv("student_data_150.csv")
train_val, test = train_test_split(data, test_size=0.15, random_state=42, stratify=data["Passed"])
y = train_val["Passed"].astype(int)
X = train_val[["StudyHours", "Attendance", "PreviousScore"]].copy()
X["WeeklyStudyHours"] = X["StudyHours"] * 7
X["AttendanceCategory"] = pd.cut(X["Attendance"], [0, 60, 80, 100], labels=["Low", "Medium", "High"], include_lowest=True).astype(str)
X["ScoreCategory"] = pd.cut(X["PreviousScore"], [0, 50, 70, 100], labels=["Low", "Medium", "High"], include_lowest=True).astype(str)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
def model(numeric, categorical=()):
    prep = ColumnTransformer([("num", StandardScaler(), list(numeric))] +
                             ([("cat", OneHotEncoder(handle_unknown="ignore"), list(categorical))] if categorical else []))
    return make_pipeline(prep, LogisticRegression(max_iter=1000))
raw = ["StudyHours", "Attendance", "PreviousScore"]
for name, num, cat in [("A: raw features", raw, ()),
                       ("A + weekly study hours", raw + ["WeeklyStudyHours"], ()),
                       ("Categories only (binned)", ["StudyHours"], ["AttendanceCategory", "ScoreCategory"]),
                       ("Raw + categories", raw, ["AttendanceCategory", "ScoreCategory"])]:
    s = cross_val_score(model(num, cat), X, y, cv=cv)
    print(f"{name:<26} CV accuracy {s.mean():.3f} +/- {s.std():.3f}")

A: raw features            CV accuracy 0.796 +/- 0.062
A + weekly study hours     CV accuracy 0.788 +/- 0.073
Categories only (binned)   CV accuracy 0.781 +/- 0.089
Raw + categories           CV accuracy 0.796 +/- 0.062


In [ ]:
# Data leakage: a feature that would not exist at prediction time. ExamScore decides pass/fail, so a "feature" built
# from it looks perfect in testing and is useless in practice (we never know it before the exam)
X_leak = X[raw].assign(ScoreChange=train_val["ExamScore"] - train_val["PreviousScore"])
for name, cols, frame in [("Raw features", raw, X), ("Raw + ScoreChange (leaks)", raw + ["ScoreChange"], X_leak)]:
    s = cross_val_score(model(cols), frame, y, cv=cv)
    print(f"{name:<26} CV accuracy {s.mean():.3f}")

Raw features               CV accuracy 0.796
Raw + ScoreChange (leaks)  CV accuracy 0.937


## 2. Categorical encoding: label, one-hot and ordinal

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Label encoding: each category gets a number, in alphabetical order
data = ["Male", "Female", "Female", "Male"]
encoder = LabelEncoder()
encoded = encoder.fit_transform(data)
print(encoded)
print(encoder.classes_)

# A target label: alphabetical order makes Fail 0 and Pass 1
results = ["Pass", "Fail", "Pass", "Fail", "Pass"]
y_encoder = LabelEncoder()
print(y_encoder.fit_transform(results), y_encoder.classes_)

[1 0 0 1]
['Female' 'Male']
[1 0 1 0 1] ['Fail' 'Pass']


In [ ]:
# One-hot encoding with pandas: one 0/1 column per city
df = pd.DataFrame({"student": ["A", "B", "C", "D"],
                   "city": ["Hyderabad", "Mumbai", "Delhi", "Hyderabad"]})
encoded_df = pd.get_dummies(df, columns=["city"], dtype=int)
print(encoded_df.to_string(index=False))      # columns come out in alphabetical order

student  city_Delhi  city_Hyderabad  city_Mumbai
      A           0               1            0
      B           0               0            1
      C           1               0            0
      D           0               1            0


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Ordinal encoding: we state the order, Low < Medium < High
levels = [["Low"], ["Medium"], ["High"], ["Medium"]]
encoder = OrdinalEncoder(categories=[["Low", "Medium", "High"]])
print(encoder.fit_transform(levels).ravel())

# Without the order, categories are sorted alphabetically: High 0, Low 1, Medium 2 (wrong order)
print(OrdinalEncoder().fit_transform(levels).ravel())

[0. 1. 2. 1.]
[1. 2. 0. 2.]


In [ ]:
# Our dataset: Class is a nominal category (no order), with 5 students recorded as "Unknown"
train_x = train_val[["StudyHours", "Attendance", "PreviousScore", "Class"]].replace({"Class": {"Unknown": np.nan}})
print(train_x["Class"].value_counts(dropna=False).to_string())
print("Pass rate by class:", y.groupby(train_x["Class"]).mean().round(2).to_dict())

# Fit the encoder on training data only; a class never seen in training ("D") becomes all zeros
onehot = OneHotEncoder(handle_unknown="ignore").fit(train_x[["Class"]].dropna())
print(onehot.categories_[0])
print(onehot.transform(pd.DataFrame({"Class": ["B", "D"]})).toarray())
try:
    OneHotEncoder().fit(train_x[["Class"]].dropna()).transform(pd.DataFrame({"Class": ["D"]}))
except ValueError as error:
    print("Without handle_unknown='ignore':", str(error)[:60])

Class
A      43
B      42
C      37
NaN     5
Pass rate by class: {'A': 0.72, 'B': 0.74, 'C': 0.68}
['A' 'B' 'C']
[[0. 1. 0.]
 [0. 0. 0.]]
Without handle_unknown='ignore': Found unknown categories ['D'] in column 0 during transform


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# A production-style pipeline: impute and scale numbers, impute and one-hot encode categories, then a model.
# Every step is fitted on the training folds only, so nothing leaks from the fold being scored
numeric_features = ["StudyHours", "Attendance", "PreviousScore"]
categorical_features = ["Class"]
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                 ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numeric_features),
                                  ("categorical", categorical_pipeline, categorical_features)])
pipe = Pipeline([("preprocessor", preprocessor),
                 ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))])
with_class = cross_val_score(pipe, train_x, y, cv=cv)
without_class = cross_val_score(pipe.set_params(preprocessor__categorical="drop"), train_x, y, cv=cv)
print(f"With Class    CV accuracy {with_class.mean():.3f}")
print(f"Without Class CV accuracy {without_class.mean():.3f}")

# Fit once on all 127 training students and score two new students, one from a class never seen
pipe.set_params(preprocessor__categorical=categorical_pipeline).fit(train_x, y)
new = pd.DataFrame({"StudyHours": [5, 5], "Attendance": [85, 85], "PreviousScore": [70, 70], "Class": ["B", "D"]})
print("P(pass):", pipe.predict_proba(new)[:, 1].round(2))

With Class    CV accuracy 0.780
Without Class CV accuracy 0.804
P(pass): [1.   0.99]
